In [ ]:
import re
import tkinter as tk
from abc import ABC, abstractmethod
from tkinter import messagebox, ttk

# ==========================================
# Domain / Business Logic Models
# ==========================================

class BankAccount(ABC):
    VALID_AGES = {
        12: "Twelve", 13: "Thirteen", 14: "Fourteen", 15: "Fifteen",
        16: "Sixteen", 17: "Seventeen", 18: "Eighteen", 19: "Nineteen",
        20: "Twenty", 21: "Twentyone", 22: "Twentytwo", 23: "Twentythree",
        24: "Twentyfour", 25: "Twentyfive", 26: "Twentysix", 27: "Twentyseven",
        28: "Twentyeight", 29: "Twentynine", 30: "Thirty", 31: "Thirtyone",
        32: "Thirtytwo", 33: "Thirtythree", 34: "Thirtyfour", 35: "Thirtyfive",
        36: "Thirtysix", 37: "Thirtyseven", 38: "Thirtyeight", 39: "Thirtynine",
        40: "Forty", 41: "Fortyone", 42: "Fortytwo", 43: "Fortythree",
        44: "Fortyfour", 45: "Fortyfive", 46: "Fortysix", 47: "Fortyseven",
        48: "Fortyeight", 49: "Fortynine", 50: "Fifty", 51: "Fiftyone",
        52: "Fiftytwo", 53: "Fiftythree", 54: "Fiftyfour", 55: "Fiftyfive",
        56: "Fiftysix", 57: "Fiftyseven", 58: "Fiftyeight", 59: "Fiftynine",
        60: "Sixty"
    }

    def __init__(self, name: str, age_num: int, age_char: str):
        if not (5 < len(name) < 20 and name == name.title()):
            raise ValueError("Name must be Title Case (e.g., S.Bruce) and 6-19 characters.")
        if not (12 <= age_num <= 60):
            raise ValueError("Age must be between 12 and 60.")
        if self.VALID_AGES.get(age_num) != age_char:
            raise ValueError(f"Age number ({age_num}) doesn't match word '{age_char}'.")

        self.name = name
        self.age_num = age_num
        self.age_char = age_char

    @abstractmethod
    def evaluate(self):
        pass


class SavingsAccount(BankAccount):
    def __init__(self, name, age_num, age_char, gender, guardian, pancard="", national_id=""):
        super().__init__(name, age_num, age_char)
        if gender.lower() not in ("male", "female"):
            raise ValueError("Gender must be 'male' or 'female'.")
        self.gender = gender.lower()
        self.guardian = guardian.lower()
        self.pancard = self._validate_pan(pancard)
        self.national_id = self._validate_id(national_id)

    def _validate_pan(self, val):
        val = val.strip().upper()
        if not re.match(r"^[A-Z]{5}[0-9]{4}[A-Z]$", val):
            raise ValueError("Invalid PAN format (Expected 5 letters, 4 digits, 1 letter).")
        return val

    def _validate_id(self, val):
        val = val.strip()
        if not re.match(r"^\d{12}$", val):
            raise ValueError("ID number must be exactly 12 digits.")
        return f"•••• •••• {val[-4:]}"

    def evaluate(self):
        is_eligible = (self.guardian == "yes" and self.age_num >= 25)
        decision = "ACTIVE / VERIFIED" if is_eligible else "DECLINED (Age < 25 or No Guardian)"
        data = {
            "Account Type": "Global Savings Tier",
            "Account Holder": self.name,
            "Customer Age": f"{self.age_num} yrs ({self.age_char})",
            "Gender": self.gender.title(),
            "PAN Identification": self.pancard,
            "National ID": self.national_id,
            "Available Balance": "₹25,000.00" if is_eligible else "₹0.00",
            "Account Identifier": "SAV-9910-4421-01" if is_eligible else "UNREGISTERED"
        }
        return is_eligible, decision, data


class JointAccount(BankAccount):
    def __init__(self, name, age_num, age_char, school_name, id_card, father_name, mother_name):
        super().__init__(name, age_num, age_char)
        if school_name != school_name.title():
            raise ValueError("School name must be in Title Case.")
        if not father_name or not mother_name:
            raise ValueError("Both parent names are required.")
        if father_name[0].upper() != mother_name[0].upper():
            raise ValueError("Father and Mother names must start with the same initial letter.")

        self.school_name = school_name
        self.id_card = id_card.lower()
        self.father_name = father_name
        self.mother_name = mother_name

    def evaluate(self):
        is_eligible = (self.id_card == "yes")
        decision = "ACTIVE / STUDENT VERIFIED" if is_eligible else "DECLINED (ID Card Required)"
        data = {
            "Account Type": "Joint Custodial Vault",
            "Primary Holder": self.name,
            "School / Institute": self.school_name,
            "Guardian 1 (Father)": self.father_name,
            "Guardian 2 (Mother)": self.mother_name,
            "Available Balance": "₹10,000.00" if is_eligible else "₹0.00",
            "Account Identifier": "JNT-2022-8114-99" if is_eligible else "UNREGISTERED"
        }
        return is_eligible, decision, data


class SalaryAccount(BankAccount):
    def __init__(self, name, age_num, age_char, company_name, grade, email_id, salary):
        super().__init__(name, age_num, age_char)
        self.company_name = company_name.strip()
        self.grade = grade.strip().upper()

        expected_email = f"{name[2:]}{self.company_name}@gmail.com"
        if email_id.strip() != expected_email:
            raise ValueError(f"Email mismatch. Expected: {expected_email}")
        self.email_id = email_id.strip()

        try:
            self.salary = float(salary)
        except ValueError:
            raise ValueError("Salary must be numeric.")
        if self.salary <= 0:
            raise ValueError("Salary must be positive.")

    def evaluate(self):
        limit = "No Pre-approved Limit"
        if self.salary >= 50000 and self.grade == "A":
            limit = "₹1.00 Cr Line"
        elif self.salary >= 50000 and self.grade == "B":
            limit = "₹10.00 L Line"
        elif self.salary >= 50000 and self.grade == "C":
            limit = "₹3.00 L Line"
        elif self.salary < 50000 and self.grade == "B":
            limit = "₹8.00 L Line"
        elif self.salary <= 30000 and self.grade == "C":
            limit = "₹1.00 L Line"

        data = {
            "Account Type": "Corporate Direct Account",
            "Account Holder": self.name,
            "Enterprise": self.company_name,
            "Corporate Grade": self.grade,
            "Corporate Email": self.email_id,
            "Monthly Inflow": f"₹{self.salary:,.2f}",
            "Lending Limit": limit,
            "Available Balance": f"₹{self.salary:,.2f}",
            "Account Identifier": "CORP-8831-2900-11"
        }
        return True, "ACTIVE CORPORATE TIER", data


class CurrentAccount(BankAccount):
    def __init__(self, name, age_num, age_char, gender, business_name, gst, revenue,
                 own_house, cibil_score, auditor, assets, f_pan, f_id, m_pan, m_id):
        super().__init__(name, age_num, age_char)
        self.business_name = business_name
        if not (len(gst) == 6 and gst[:3].isalpha() and gst[3:].isdigit()):
            raise ValueError("GST must be 3 letters + 3 numbers (e.g., IND123).")
        self.gst = gst.upper()
        self.revenue = revenue
        self.own_house = own_house.lower()
        if cibil_score.upper() != "A":
            raise ValueError("CIBIL rating grade must be 'A'.")
        self.cibil_score = cibil_score.upper()
        self.auditor = auditor.lower()
        if assets not in ("1cr", "2cr", "3cr"):
            raise ValueError("Assets must be: 1cr, 2cr, or 3cr.")
        self.assets = assets

    def evaluate(self):
        eligible = (self.own_house == "yes" and self.auditor == "yes")
        decision = "VERIFIED ENTERPRISE" if eligible else "PENDING AUDIT/PROPERTY"
        data = {
            "Account Type": "Commercial Vault",
            "Business Name": self.business_name,
            "Signatory": self.name,
            "GSTIN": self.gst,
            "Declared Revenue": f"₹{self.revenue}",
            "Capital Assets": self.assets.upper(),
            "Overdraft Facility": "₹50,00,000.00" if eligible else "Unavailable",
            "Available Balance": "₹1,50,000.00" if eligible else "₹0.00",
            "Account Identifier": "COM-0199-5501-72" if eligible else "UNREGISTERED"
        }
        return eligible, decision, data


# ==========================================
# Pure Standard Tkinter Neo-Bank GUI
# ==========================================

class ModernStandardBankApp(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Nexus Digital Banking")
        self.geometry("1050x700")
        self.minsize(950, 650)
        self.configure(bg="#0F172A")  # Deep modern navy/slate

        # Global Color Palette
        self.C_BG = "#0F172A"
        self.C_SIDEBAR = "#1E293B"
        self.C_CARD = "#1E293B"
        self.C_INPUT = "#334155"
        self.C_ACCENT = "#38BDF8"
        self.C_TEXT = "#F8FAFC"
        self.C_MUTED = "#94A3B8"

        self.account_type_var = tk.StringVar(value="Savings")
        self.form_entries = {}

        self._init_layout()
        self.show_application_page()

    def _init_layout(self):
        self.grid_columnconfigure(1, weight=1)
        self.grid_rowconfigure(0, weight=1)

        # Persistent Modern Sidebar
        self.sidebar = tk.Frame(self, bg=self.C_SIDEBAR, width=240)
        self.sidebar.grid(row=0, column=0, sticky="nsew")
        self.sidebar.grid_propagate(False)

        # Brand header
        tk.Label(self.sidebar, text="⚡ NEXUS", font=("Helvetica", 20, "bold"), fg=self.C_ACCENT, bg=self.C_SIDEBAR).pack(anchor="w", padx=25, pady=(30, 25))

        tk.Label(self.sidebar, text="ACCOUNT TIERS", font=("Helvetica", 9, "bold"), fg=self.C_MUTED, bg=self.C_SIDEBAR).pack(anchor="w", padx=25, pady=(0, 8))

        tiers = ["Savings", "Joint", "Salary", "Current"]
        for tier in tiers:
            btn = tk.Button(
                self.sidebar,
                text=f"  {tier} Tier",
                font=("Helvetica", 11),
                fg=self.C_TEXT,
                bg=self.C_SIDEBAR,
                activebackground=self.C_INPUT,
                activeforeground=self.C_ACCENT,
                bd=0,
                cursor="hand2",
                anchor="w",
                padx=15,
                pady=10,
                command=lambda t=tier: self._on_tier_selected(t)
            )
            btn.pack(fill="x", padx=15, pady=2)

        # Main Dynamic View
        self.main_view = tk.Frame(self, bg=self.C_BG)
        self.main_view.grid(row=0, column=1, sticky="nsew")

    def _on_tier_selected(self, tier):
        self.account_type_var.set(tier)
        self.show_application_page()

    def _clear_main_view(self):
        for widget in self.main_view.winfo_children():
            widget.destroy()

    # ----------------------------------------------------
    # PAGE 1: Modern Dark Form
    # ----------------------------------------------------
    def show_application_page(self):
        self._clear_main_view()
        self.form_entries.clear()

        header = tk.Frame(self.main_view, bg=self.C_BG)
        header.pack(fill="x", padx=40, pady=(35, 10))

        tk.Label(header, text="Account Onboarding Engine", font=("Helvetica", 22, "bold"), fg=self.C_TEXT, bg=self.C_BG).pack(anchor="w")
        tk.Label(header, text=f"Currently configuring: {self.account_type_var.get()} Profile", font=("Helvetica", 11), fg=self.C_MUTED, bg=self.C_BG).pack(anchor="w", pady=(3, 0))

        # Main Card Panel
        form_card = tk.Frame(self.main_view, bg=self.C_CARD, padx=30, pady=25)
        form_card.pack(fill="both", expand=True, padx=40, pady=15)

        # Fields definition
        acc_type = self.account_type_var.get()
        fields = [
            ("Full Legal Name", "name", "S.Bruce"),
            ("Numeric Age (12-60)", "age_num", "26"),
            ("Age Spelled Out", "age_char", "Twentysix"),
        ]

        if acc_type == "Savings":
            fields += [
                ("Gender (male/female)", "gender", "male"),
                ("Guardian Verification (yes/no)", "guardian", "yes"),
                ("PAN Card Number", "pancard", "ABCDE1234F"),
                ("12-Digit Identification", "national_id", "123456789123"),
            ]
        elif acc_type == "Joint":
            fields += [
                ("School / University Name", "school_name", "Bat School"),
                ("ID Card Verified (yes/no)", "id_card", "yes"),
                ("Father's Name", "father_name", "Bruce Wayne"),
                ("Mother's Name (Same Initial)", "mother_name", "B.Catwomen"),
            ]
        elif acc_type == "Salary":
            fields += [
                ("Corporate Employer", "company_name", "infosys"),
                ("Employee Grade (A/B/C)", "grade", "B"),
                ("Corporate Email Address", "email_id", "Bruceinfosys@gmail.com"),
                ("Base Monthly Salary (₹)", "salary", "35000"),
            ]
        else:  # Current
            fields += [
                ("Registered Company Trade", "business_name", "Bruce Corporation"),
                ("GST Registration Number", "gst", "IND123"),
                ("Annual Revenue Inflow (₹)", "revenue", "1000000"),
                ("Owned Real Estate (yes/no)", "own_house", "yes"),
                ("CIBIL Grade (A)", "cibil_score", "A"),
                ("Auditor Clearance (yes/no)", "auditor", "yes"),
                ("Capital Assets (1cr/2cr/3cr)", "assets", "3cr"),
            ]

        for i, (label_text, key, default_val) in enumerate(fields):
            row_frame = tk.Frame(form_card, bg=self.C_CARD)
            row_frame.pack(fill="x", pady=6)

            lbl = tk.Label(row_frame, text=label_text, font=("Helvetica", 10, "bold"), fg=self.C_TEXT, bg=self.C_CARD, width=28, anchor="w")
            lbl.pack(side="left")

            entry = tk.Entry(row_frame, font=("Helvetica", 11), bg=self.C_INPUT, fg=self.C_TEXT, insertbackground="white", bd=0, relief="flat")
            entry.insert(0, default_val)
            entry.pack(side="left", fill="x", expand=True, ipady=6, padx=(10, 0))
            self.form_entries[key] = entry

        # Bottom Action Bar
        action_bar = tk.Frame(self.main_view, bg=self.C_BG)
        action_bar.pack(fill="x", padx=40, pady=(0, 25))

        btn = tk.Button(
            action_bar,
            text="Verify & Open Account ➔",
            font=("Helvetica", 11, "bold"),
            bg="#2563EB",
            fg="white",
            activebackground="#1D4ED8",
            activeforeground="white",
            bd=0,
            padx=20,
            pady=10,
            cursor="hand2",
            command=self._handle_submission
        )
        btn.pack(side="right")

    def _handle_submission(self):
        vals = {k: v.get().strip() for k, v in self.form_entries.items()}
        acc_type = self.account_type_var.get()

        try:
            name = vals.get("name", "")
            age_num = int(vals.get("age_num", 0))
            age_char = vals.get("age_char", "")

            if acc_type == "Savings":
                account = SavingsAccount(name, age_num, age_char, vals["gender"], vals["guardian"], vals["pancard"], vals["national_id"])
            elif acc_type == "Joint":
                account = JointAccount(name, age_num, age_char, vals["school_name"], vals["id_card"], vals["father_name"], vals["mother_name"])
            elif acc_type == "Salary":
                account = SalaryAccount(name, age_num, age_char, vals["company_name"], vals["grade"], vals["email_id"], vals["salary"])
            else:
                account = CurrentAccount(name, age_num, age_char, "male", vals["business_name"], vals["gst"], vals["revenue"],
                                         vals["own_house"], vals["cibil_score"], vals["auditor"], vals["assets"],
                                         "ABCDE1234F", "123456789123", "EDCBA1234F", "987654321987")

            is_eligible, status_msg, account_data = account.evaluate()
            self.show_dashboard_page(account_data, status_msg, is_eligible)

        except ValueError as err:
            messagebox.showerror("Verification Notice", str(err))

    # ----------------------------------------------------
    # PAGE 2: Modern Dark Bank Dashboard
    # ----------------------------------------------------
    def show_dashboard_page(self, account_data, status_msg, is_eligible):
        self._clear_main_view()

        # Dashboard Top Navigation
        top_nav = tk.Frame(self.main_view, bg=self.C_BG)
        top_nav.pack(fill="x", padx=40, pady=(30, 15))

        user_name = account_data.get("Account Holder", account_data.get("Primary Holder", account_data.get("Signatory", "Client")))
        tk.Label(top_nav, text=f"Welcome back, {user_name}", font=("Helvetica", 20, "bold"), fg=self.C_TEXT, bg=self.C_BG).pack(side="left")

        # Pill Status Badge
        badge_bg = "#10B981" if is_eligible else "#EF4444"
        badge = tk.Label(top_nav, text=f"● {status_msg}", font=("Helvetica", 10, "bold"), fg="white", bg=badge_bg, padx=12, pady=5)
        badge.pack(side="right")

        # Split Container
        content = tk.Frame(self.main_view, bg=self.C_BG)
        content.pack(fill="both", expand=True, padx=40, pady=10)
        content.grid_columnconfigure(0, weight=4)
        content.grid_columnconfigure(1, weight=6)

        # LEFT COLUMN: Cards
        left_col = tk.Frame(content, bg=self.C_BG)
        left_col.grid(row=0, column=0, sticky="nsew", padx=(0, 20))

        # Modern Frosted Dark Debit Card
        card = tk.Frame(left_col, bg="#0B132B", height=180, padx=20, pady=15, relief="flat", highlightbackground="#334155", highlightthickness=1)
        card.pack(fill="x", pady=(0, 15))
        card.pack_propagate(False)

        tk.Label(card, text="NEXUS PLATINUM", font=("Helvetica", 10, "bold"), fg=self.C_ACCENT, bg="#0B132B").pack(anchor="w")
        tk.Label(card, text="••••  ••••  ••••  9821", font=("Courier", 16, "bold"), fg="white", bg="#0B132B").pack(anchor="w", pady=(25, 10))

        card_footer = tk.Frame(card, bg="#0B132B")
        card_footer.pack(fill="x", side="bottom")
        tk.Label(card_footer, text=user_name.upper(), font=("Helvetica", 10, "bold"), fg=self.C_TEXT, bg="#0B132B").pack(side="left")
        tk.Label(card_footer, text="EXP 09/29", font=("Helvetica", 9), fg=self.C_MUTED, bg="#0B132B").pack(side="right")

        # Balance Card
        bal_box = tk.Frame(left_col, bg=self.C_CARD, padx=20, pady=15)
        bal_box.pack(fill="x")
        tk.Label(bal_box, text="Available Liquid Balance", font=("Helvetica", 10), fg=self.C_MUTED, bg=self.C_CARD).pack(anchor="w")
        tk.Label(bal_box, text=account_data.get("Available Balance", "₹0.00"), font=("Helvetica", 22, "bold"), fg="#10B981", bg=self.C_CARD).pack(anchor="w", pady=(3, 2))
        tk.Label(bal_box, text=f"Routing ID: {account_data.get('Account Identifier', 'N/A')}", font=("Courier", 9), fg=self.C_MUTED, bg=self.C_CARD).pack(anchor="w")

        # RIGHT COLUMN: Portfolio Specs
        right_col = tk.Frame(content, bg=self.C_CARD, padx=25, pady=20)
        right_col.grid(row=0, column=1, sticky="nsew")

        tk.Label(right_col, text="Account Matrix Details", font=("Helvetica", 13, "bold"), fg=self.C_TEXT, bg=self.C_CARD).pack(anchor="w", pady=(0, 15))

        for title, val in account_data.items():
            row = tk.Frame(right_col, bg=self.C_CARD)
            row.pack(fill="x", pady=6)
            tk.Label(row, text=title, font=("Helvetica", 10), fg=self.C_MUTED, bg=self.C_CARD).pack(side="left")
            tk.Label(row, text=str(val), font=("Helvetica", 10, "bold"), fg=self.C_TEXT, bg=self.C_CARD).pack(side="right")

        # Bottom Return Button
        bottom_bar = tk.Frame(self.main_view, bg=self.C_BG)
        bottom_bar.pack(fill="x", padx=40, pady=(10, 25))

        back_btn = tk.Button(
            bottom_bar,
            text="← Switch Tier",
            font=("Helvetica", 10, "bold"),
            bg=self.C_CARD,
            fg=self.C_TEXT,
            activebackground=self.C_INPUT,
            activeforeground="white",
            bd=0,
            padx=15,
            pady=8,
            cursor="hand2",
            command=self.show_application_page
        )
        back_btn.pack(side="left")


if __name__ == "__main__":
    app = ModernStandardBankApp()
    app.mainloop()